In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else "cpu"

## 1. Reproducibility

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root.")

PROJECT_ROOT = find_project_root()
REPORT_DIR = PROJECT_ROOT / "reports" / "damage_no_damage_evaluation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

## 2. Selecting the damage checkpoint

In [ ]:
DAMAGE_CHECKPOINT = (
    PROJECT_ROOT
    / "models"
    / "damage_hyperparameter_tuning"
    / "640_longer"
    / "weights"
    / "best.pt"
)

print("Damage checkpoint:", DAMAGE_CHECKPOINT)

if not DAMAGE_CHECKPOINT.exists():
    raise FileNotFoundError(
        "Update DAMAGE_CHECKPOINT to the selected damage model."
    )

model = YOLO(str(DAMAGE_CHECKPOINT))

## 3. Locating CompCars modelling CSV

In [ ]:
CSV_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "compcars_modeling_ready.csv",
    PROJECT_ROOT / "compcars_modeling_ready.csv",
    Path(
        r"G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J"
        r"\capstone-vehicle-metadata\outputs\compcars_modeling_ready.csv"
    ),
]

COMPCARS_CSV = next(
    (p for p in CSV_CANDIDATES if p.exists()),
    None,
)

print("COMPCARS_CSV:", COMPCARS_CSV)

if COMPCARS_CSV is None:
    raise FileNotFoundError(
        "Could not find compcars_modeling_ready.csv. "
        "Set COMPCARS_CSV manually."
    )

## 4. Resolving image paths

In [ ]:
df = pd.read_csv(COMPCARS_CSV)

COMPCARS_ROOT_CANDIDATES = [
    Path(
        r"G:\.shortcut-targets-by-id\1OOvt005TduvEA8h6_snfuyQtbQAjo-3J"
        r"\capstone-vehicle-metadata\data\compcars_kaggle\extracted"
    ),
    PROJECT_ROOT / "data" / "raw" / "compcars_kaggle" / "extracted",
]

COMPCARS_ROOT = next(
    (p for p in COMPCARS_ROOT_CANDIDATES if p.exists()),
    None,
)

print("COMPCARS_ROOT:", COMPCARS_ROOT)

def resolve_path(raw):
    raw = str(raw)
    direct = Path(raw)

    if direct.exists():
        return direct

    normalized = raw.replace("\\", "/")

    if "/image/" in normalized and COMPCARS_ROOT is not None:
        suffix = normalized.split("/image/", 1)[1]

        return (
            COMPCARS_ROOT
            / "image"
            / Path(*suffix.split("/"))
        )

    return direct

df["resolved_path"] = df["file_path"].map(resolve_path)

exists_mask = df["resolved_path"].map(Path.exists)

print("Resolved:", int(exists_mask.sum()), "/", len(df))

df = df.loc[exists_mask].copy()

## 5. Sampling ordinary vehicle images

In [ ]:
SAMPLE_SIZE = 500

sample_df = (
    df.sample(
        n=min(SAMPLE_SIZE, len(df)),
        random_state=SEED,
    )
    .reset_index(drop=True)
)

print("Sample size:", len(sample_df))

## 6. Running confidence thresholds

In [ ]:
CONF_THRESHOLDS = [0.25, 0.35, 0.50]
IMGSZ = 640

summary_counts = {
    threshold: {
        "images_with_prediction": 0,
        "total_predictions": 0,
    }
    for threshold in CONF_THRESHOLDS
}

prediction_rows = []

image_paths = [
    Path(p)
    for p in sample_df["resolved_path"]
]

print("Images to evaluate:", len(image_paths))

for i, image_path in enumerate(image_paths):

    result = model.predict(
        source=str(image_path),
        imgsz=IMGSZ,
        conf=min(CONF_THRESHOLDS),
        device=DEVICE,
        verbose=False,
    )[0]

    boxes = result.boxes

    if boxes is not None and len(boxes) > 0:

        cls_ids = boxes.cls.cpu().numpy().astype(int)
        confidences = boxes.conf.cpu().numpy()

        # Reuse the same predictions for every threshold.
        for threshold in CONF_THRESHOLDS:

            keep = confidences >= threshold

            num_predictions = int(keep.sum())

            if num_predictions > 0:
                summary_counts[threshold]["images_with_prediction"] += 1
                summary_counts[threshold]["total_predictions"] += num_predictions

                for cls_id, confidence in zip(
                    cls_ids[keep],
                    confidences[keep],
                ):
                    prediction_rows.append({
                        "confidence_threshold": threshold,
                        "resolved_path": str(image_path),
                        "predicted_damage": model.names[int(cls_id)],
                        "prediction_confidence": float(confidence),
                    })

    del result

    if torch.cuda.is_available() and (i + 1) % 50 == 0:
        torch.cuda.empty_cache()

    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(image_paths)} images")


summary_rows = []

for threshold in CONF_THRESHOLDS:

    images_with_prediction = (
        summary_counts[threshold]["images_with_prediction"]
    )

    total_predictions = (
        summary_counts[threshold]["total_predictions"]
    )

    summary_rows.append({
        "confidence_threshold": threshold,
        "images_evaluated": len(image_paths),
        "images_with_any_damage_prediction": images_with_prediction,
        "prediction_image_rate": (
            images_with_prediction / len(image_paths)
        ),
        "total_damage_predictions": total_predictions,
    })


summary_df = pd.DataFrame(summary_rows)
prediction_df = pd.DataFrame(prediction_rows)

display(summary_df)

summary_df.to_csv(
    REPORT_DIR / "normal_vehicle_prediction_rate_by_threshold.csv",
    index=False,
)

prediction_df.to_csv(
    REPORT_DIR / "normal_vehicle_damage_predictions.csv",
    index=False,
)

## 7. Displaying highest-confidence predicted cases

In [ ]:
if len(prediction_df) == 0:
    print("No damage predictions were produced.")
else:
    review_df = (
        prediction_df[
            prediction_df["confidence_threshold"] == 0.25
        ]
        .sort_values(
            "prediction_confidence",
            ascending=False,
        )
        .drop_duplicates("resolved_path")
        .head(30)
    )

    print("Images to review:", len(review_df))

    for _, row in review_df.iterrows():
        image_path = row["resolved_path"]

        result = model.predict(
            source=image_path,
            imgsz=IMGSZ,
            conf=0.25,
            device=DEVICE,
            verbose=False,
        )[0]

        plotted = result.plot()[:, :, ::-1]

        plt.figure(figsize=(10, 7))
        plt.imshow(plotted)
        plt.title(
            f"{Path(image_path).name} | "
            f"{row['predicted_damage']} "
            f"{row['prediction_confidence']:.2f}"
        )
        plt.axis("off")
        plt.tight_layout()
        plt.show()